# MANTA Full Google Colab Pipeline (with Top-5 Checkpoints + Resume)

This notebook runs the requested end-to-end flow:
1. Download 50 real stars (quarters 1,2,3).
2. Run one known target through preprocessing + decomposition manually.
3. Auto-answer diagnostic physics questions.
4. Run one full MANTA forward pass and verify output bounds.
5. Train with top-5 checkpoint retention and resume-from-top5 support.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/YOUR_ORG/manta.git'  # TODO: replace once
PROJECT_DIR = Path('/content/manta')

if PROJECT_DIR.exists():
    os.chdir(PROJECT_DIR)
    print('Using existing repo at', PROJECT_DIR)
    try:
        subprocess.check_call(['git', 'fetch', 'origin'])
        subprocess.check_call(['git', 'pull', '--ff-only', 'origin', 'main'])
        print('Repo synced to latest origin/main')
    except Exception as e:
        print('Warning: could not auto-pull latest main:', e)
else:
    if 'YOUR_ORG' in REPO_URL:
        raise ValueError('Set REPO_URL in this cell to your GitHub repo before running clone.')
    print('Repo not found at /content/manta. Cloning now...')
    subprocess.check_call(['git', 'clone', REPO_URL, '/content/manta'])
    os.chdir(PROJECT_DIR)

print('Current working directory:', Path.cwd())
print('Python version:', sys.version.split()[0])

In [ ]:
# Install dependencies robustly in Colab (no local package install)
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path('/content/manta')
if ROOT.exists():
    os.chdir(ROOT)

if not Path('requirements.txt').exists():
    raise FileNotFoundError('requirements.txt not found. Ensure the repo setup cell succeeded.')

def run(cmd):
    print('+', ' '.join(cmd))
    subprocess.check_call(cmd)

# Keep packaging tools fresh
run([sys.executable, '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel'])

# Install runtime dependencies only
run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])

# Optional compatibility pin for Colab preinstalled datasets/gcsfs
run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'fsspec==2025.3.0'])

# Make local repo importable without pip install . or pip install -e .
repo_path = str(Path.cwd())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
os.environ['PYTHONPATH'] = repo_path + (':' + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else '')

print('Dependency installation complete.')
print('Local package install intentionally skipped in Colab.')
print('Working directory:', Path.cwd())
print('PYTHONPATH:', os.environ['PYTHONPATH'])

In [ ]:
# Quick sanity check: repo imports should work without local package install
import os
import sys
from pathlib import Path

os.chdir('/content/manta')
repo_path = str(Path.cwd())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import manta  # noqa: F401
print('manta import OK from', repo_path)

## Create Colab Config
This writes `configs/manta_colab.yaml` and uses Drive paths for outputs/checkpoints.

In [ ]:
import yaml

base_cfg_path = Path('configs/manta_default.yaml')
cfg = yaml.safe_load(base_cfg_path.read_text(encoding='utf-8'))

cfg['device'] = 'cuda'
cfg['data']['cache_dir'] = '/content/drive/MyDrive/manta/data/cache'
cfg['data']['diagnostics_dir'] = '/content/drive/MyDrive/manta/outputs/diagnostics'
cfg['output_dir'] = '/content/drive/MyDrive/manta/outputs'
cfg['training']['checkpoint_dir'] = '/content/drive/MyDrive/manta/checkpoints'
cfg['training']['num_workers'] = 2
cfg['data']['max_download_workers'] = 4

colab_cfg = Path('configs/manta_colab.yaml')
colab_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('Wrote', colab_cfg)

## Step 1: Download 50 real stars (quarters 1, 2, 3)
Requested command (now supported by script):
`python scripts/download_data.py --cache_dir ./data/cache --n_stars 50 --quarters 1 2 3`

In [ ]:
# Optional once per runtime: clear stale/corrupted Lightkurve FITS cache
!rm -rf /root/.lightkurve/cache/mastDownload/Kepler

!PYTHONPATH=. python scripts/download_data.py \
  --config configs/manta_colab.yaml \
  --cache_dir ./data/cache \
  --n_stars 50 \
  --quarters 1 2 3 \
  --max-workers 2

## Step 2: Run one star through full pipeline manually

In [ ]:
from IPython.display import Image, display
import numpy as np

from manta.data.downloader import download_lightcurve
from manta.data.preprocessor import PreprocessingPipeline
from manta.data.frequency_decomposer import FrequencyDecomposer

# Known confirmed host (as requested)
lc = download_lightcurve(kepler_id=757450, quarter=1, cache_dir='./data/cache/raw')

pipeline = PreprocessingPipeline()
result = pipeline.fit_transform(lc)

decomposer = FrequencyDecomposer(diagnostics_dir='outputs/diagnostics')
bands = decomposer.decompose(
    result['final_flux'],
    result['final_time'],
    cadence_days=0.0204,
)

plot_path = decomposer.plot_decomposition(result['final_flux'], result['final_time'], bands, filename_stem='colab_manual_star')
print('Saved decomposition plot:', plot_path)
display(Image(filename=str(plot_path)))

## Step 3: Diagnostic questions and automatic answers

In [ ]:
import numpy as np

reconstructed = decomposer.reconstruct(bands)
reconstruction_error = float(np.max(np.abs(result['final_flux'] - reconstructed)))

has_nan = any([
    np.isnan(bands.granulation).any(),
    np.isnan(bands.asteroseismology).any(),
    np.isnan(bands.starspot).any(),
])

gran_diff_std = float(np.std(np.diff(bands.granulation)))
astero_diff_std = float(np.std(np.diff(bands.asteroseismology)))
gran_smooth = gran_diff_std < astero_diff_std

astero_amp = float(np.std(bands.asteroseismology))
gran_amp = float(np.std(bands.granulation))
astero_rapid_low_amp = (astero_diff_std > gran_diff_std) and (astero_amp < gran_amp * 1.5)

def dominant_period_days(signal: np.ndarray, cadence_days: float = 0.0204) -> float:
    centered = signal - np.mean(signal)
    fft_vals = np.fft.rfft(centered)
    freqs = np.fft.rfftfreq(centered.size, d=cadence_days)
    if freqs.size <= 1:
        return float('nan')
    freqs = freqs[1:]
    power = np.abs(fft_vals[1:])
    if np.all(power <= 0):
        return float('nan')
    f_peak = float(freqs[int(np.argmax(power))])
    return float('inf') if f_peak <= 0 else 1.0 / f_peak

starspot_period_days = dominant_period_days(bands.starspot)
starspot_periodic = np.isfinite(starspot_period_days) and (0.5 <= starspot_period_days <= 60.0)

print('Q1. Does the granulation band look like slow smooth variation?')
print('   ->', 'YES' if gran_smooth else 'NO', f'(gran diff std={gran_diff_std:.3e}, astero diff std={astero_diff_std:.3e})')

print('Q2. Does the asteroseismology band look like rapid low-amplitude oscillations?')
print('   ->', 'YES' if astero_rapid_low_amp else 'NO', f'(astero std={astero_amp:.3e}, gran std={gran_amp:.3e})')

print('Q3. Does the starspot band show clear periodicity?')
print('   ->', 'YES' if starspot_periodic else 'NO', f'(dominant period ~ {starspot_period_days:.2f} days)')

print('Q4. Is reconstruction error below 1e-4?')
print('   ->', 'YES' if reconstruction_error < 1e-4 else 'NO', f'(max abs error={reconstruction_error:.3e})')

print('Q5. Are there any NaN values surviving into the bands?')
print('   ->', 'NO' if not has_nan else 'YES')

## Step 4: Run one full forward pass

In [ ]:
import torch
from manta.models.manta import MANTA
from manta.utils.config import load_config

config = load_config('configs/manta_colab.yaml')
model = MANTA(config)
print(model.get_parameter_count())

batch = {
    'global_view': torch.randn(4, 1, 2001),
    'local_view': torch.randn(4, 1, 201),
    'freq_bands': torch.randn(4, 3, 2001),
    'stellar_params': torch.randn(4, 5),
    'label': torch.randint(0, 2, (4,)).float(),
}

model.eval()
with torch.no_grad():
    out = model(batch)

print('Output shape:', out.shape)
print('Output min/max:', out.min().item(), out.max().item())

## Step 5: Build train/val loaders for 50-star subset
This filters to stars and quarters that are already downloaded in Step 1 so training does not unexpectedly trigger large extra downloads.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader

from manta.utils.config import load_config
from manta.data.downloader import download_kepler_tce_catalog
from manta.data.dataset import KeplerTransitDataset, split_dataset

REQUESTED_QUARTERS = [1, 2, 3]
N_STARS = 50

config = load_config('configs/manta_colab.yaml')
config.data.cache_dir = './data/cache'

tce_df = download_kepler_tce_catalog(cache_dir=config.data.cache_dir, url=config.data.tce_catalog_url)

if 'kepid' in tce_df.columns:
    kepid_col = 'kepid'
elif 'kepler_id' in tce_df.columns:
    kepid_col = 'kepler_id'
else:
    raise KeyError('No Kepler ID column found (expected kepid or kepler_id).')

quarter_col = None
for candidate in ('quarter', 'koi_quarter'):
    if candidate in tce_df.columns:
        quarter_col = candidate
        break

if quarter_col is not None:
    candidate_rows = tce_df[tce_df[quarter_col].isin(REQUESTED_QUARTERS)].copy()
else:
    candidate_rows = tce_df.copy()
    quarter_col = 'quarter'
    print('No quarter column in catalog; expanding selected rows across requested quarters:', REQUESTED_QUARTERS)

stars = candidate_rows[kepid_col].dropna().astype(int).unique()
if len(stars) == 0:
    raise ValueError('No stars available after initial filtering.')

rng = np.random.default_rng(42)
picked = set(rng.choice(stars, size=min(N_STARS, len(stars)), replace=False).tolist())
subset_base = candidate_rows[candidate_rows[kepid_col].astype(int).isin(picked)].copy()

if quarter_col not in tce_df.columns:
    q_df = pd.DataFrame({quarter_col: REQUESTED_QUARTERS})
    subset = subset_base.assign(_join_key=1).merge(q_df.assign(_join_key=1), on='_join_key').drop(columns=['_join_key'])
else:
    subset = subset_base

raw_cache = Path(config.data.cache_dir) / 'raw'
def is_cached(row):
    path = raw_cache / f"kic_{int(row[kepid_col])}_q{int(row[quarter_col])}.pkl"
    return path.exists()

subset = subset[subset.apply(is_cached, axis=1)].copy()
print('Rows after cache filter:', len(subset), '| unique stars:', subset[kepid_col].nunique())

dataset = KeplerTransitDataset(
    tce_catalog=subset,
    cache_dir=config.data.cache_dir,
    preprocessing_config={
        'nan_strategy': config.data.nan_strategy,
        'normalization_method': config.data.normalization_method,
        'sigma_clip_threshold': config.data.sigma_clip_threshold,
        'global_view_bins': config.data.global_view_bins,
        'local_view_bins': config.data.local_view_bins,
        'kepler_cadence_days': config.data.kepler_cadence_days,
        'diagnostics_dir': config.data.diagnostics_dir,
    },
    augmentation_config={'use_augmentation': config.augmentation.use_augmentation},
)

train_ds, val_ds, test_ds = split_dataset(dataset, train_ratio=0.8, val_ratio=0.1, seed=config.seed)

train_loader = DataLoader(train_ds, batch_size=config.training.batch_size, shuffle=True, num_workers=config.training.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=config.training.batch_size, shuffle=False, num_workers=config.training.num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=config.training.batch_size, shuffle=False, num_workers=config.training.num_workers, pin_memory=True)

print('Train/Val/Test sizes:', len(train_ds), len(val_ds), len(test_ds))

## Step 6: Configure trainer + Top-5 checkpoint manager

In [ ]:
import json
import os
from pathlib import Path

import torch
from torch.optim import AdamW

from manta.models.manta import MANTA
from manta.training.loss import FocalBCELoss
from manta.training.scheduler import PhysicsWarmupScheduler
from manta.training.trainer import MANTATrainer

device = 'cuda' if torch.cuda.is_available() else 'cpu'

checkpoint_root = Path('/content/drive/MyDrive/manta/checkpoints_top5')
checkpoint_root.mkdir(parents=True, exist_ok=True)
topk_index_path = checkpoint_root / 'top5_index.json'
TOPK = 5

model = MANTA(config)
optimizer = AdamW(model.parameters(), lr=config.training.learning_rate, weight_decay=config.training.weight_decay)
scheduler = PhysicsWarmupScheduler(
    optimizer=optimizer,
    warmup_epochs=config.training.warmup_epochs,
    total_epochs=config.training.max_epochs,
    min_lr=config.training.min_lr,
)
loss_fn = FocalBCELoss(gamma=config.training.focal_gamma, alpha=config.training.focal_alpha)

trainer = MANTATrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_fn=loss_fn,
    device=device,
    config=config,
    checkpoint_dir=checkpoint_root,
)

def _load_topk_index():
    if topk_index_path.exists():
        return json.loads(topk_index_path.read_text(encoding='utf-8'))
    return []

def _save_topk_index(index_rows):
    topk_index_path.write_text(json.dumps(index_rows, indent=2), encoding='utf-8')

def update_topk(epoch, val_auc, checkpoint_path, topk=TOPK):
    rows = _load_topk_index()
    rows.append({
        'epoch': int(epoch),
        'val_auc_roc': float(val_auc),
        'path': str(checkpoint_path),
    })
    rows = sorted(rows, key=lambda r: r['val_auc_roc'], reverse=True)
    keep = rows[:topk]
    drop = rows[topk:]

    for rec in drop:
        p = Path(rec['path'])
        if p.exists():
            p.unlink()
        m = p.parent / f"metrics_{p.stem.replace('checkpoint_', '')}.json"
        if m.exists():
            m.unlink()

    _save_topk_index(keep)
    return keep

def list_top5():
    rows = _load_topk_index()
    if not rows:
        print('No checkpoints in top-5 index yet.')
        return rows
    for i, rec in enumerate(rows, start=1):
        print(f"#{i} | epoch={rec['epoch']} | val_auc={rec['val_auc_roc']:.5f} | {rec['path']}")
    return rows

def resume_from_top5(rank=1):
    rows = _load_topk_index()
    if not rows:
        raise ValueError('Top-5 list is empty; no checkpoint to resume from.')
    if rank < 1 or rank > len(rows):
        raise ValueError(f'Rank must be between 1 and {len(rows)}')

    ckpt_path = rows[rank - 1]['path']
    payload = trainer.load_checkpoint(ckpt_path)
    next_epoch = int(payload['epoch']) + 1
    print(f"Resumed from rank #{rank} checkpoint: {ckpt_path}")
    print(f"Next epoch: {next_epoch}")
    return next_epoch

list_top5()

## Step 7: Train and keep best 5 checkpoints
Set `MAX_EPOCHS` as needed. You can interrupt and later resume from top-5.

In [ ]:
MAX_EPOCHS = 10  # set to 80 for full training
start_epoch = 1
history = []

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    train_metrics = trainer.train_epoch(train_loader)
    val_metrics = trainer.validate(val_loader)

    epoch_metrics = {
        'epoch': float(epoch),
        **{f'train_{k}': float(v) for k, v in train_metrics.items()},
        **{f'val_{k}': float(v) for k, v in val_metrics.items()},
    }
    history.append(epoch_metrics)

    ckpt_path = trainer.save_checkpoint(epoch=epoch, metrics=epoch_metrics, tag=f'epoch{epoch:03d}')
    top5 = update_topk(epoch=epoch, val_auc=float(val_metrics['auc_roc']), checkpoint_path=ckpt_path, topk=TOPK)

    print(
        f"Epoch {epoch:03d} | train_loss={train_metrics['loss']:.5f} | val_auc={val_metrics['auc_roc']:.5f} | top5_kept={len(top5)}"
    )

print('Training chunk complete.')
list_top5()

## Step 8: Restart from one of the saved top-5 checkpoints
Set `RESUME_RANK` from 1..5, then continue training for more epochs.

In [ ]:
RESUME_RANK = 1  # choose 1..5 based on list_top5()
start_epoch = resume_from_top5(rank=RESUME_RANK)
EXTRA_EPOCHS = 5
end_epoch = start_epoch + EXTRA_EPOCHS - 1

for epoch in range(start_epoch, end_epoch + 1):
    train_metrics = trainer.train_epoch(train_loader)
    val_metrics = trainer.validate(val_loader)

    epoch_metrics = {
        'epoch': float(epoch),
        **{f'train_{k}': float(v) for k, v in train_metrics.items()},
        **{f'val_{k}': float(v) for k, v in val_metrics.items()},
    }

    ckpt_path = trainer.save_checkpoint(epoch=epoch, metrics=epoch_metrics, tag=f'epoch{epoch:03d}')
    update_topk(epoch=epoch, val_auc=float(val_metrics['auc_roc']), checkpoint_path=ckpt_path, topk=TOPK)

    print(f"Resumed epoch {epoch:03d} | val_auc={val_metrics['auc_roc']:.5f}")

print('Resume training chunk complete.')
list_top5()

## Optional: Evaluate best checkpoint

In [ ]:
rows = list_top5()
if rows:
    best_ckpt = rows[0]['path']
    print('Evaluating best checkpoint:', best_ckpt)
    !PYTHONPATH=. python scripts/evaluate.py --config configs/manta_colab.yaml --model manta --checkpoint "{best_ckpt}" --output-dir /content/drive/MyDrive/manta/eval
else:
    print('No checkpoints available to evaluate yet.')